In [1]:
!pip install yfinance

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf

In [3]:
%pip install --quiet numpy matplotlib scipy pandas scikit-learn yfinance

import os
import sys
import importlib
from pathlib import Path

repo_path = "/content/Quant-Finance-Summer-2026"

if not os.path.isdir(os.path.join(repo_path, ".git")):
    !git clone https://github.com/munawarali93/Quant-Finance-Summer-2026.git "$repo_path"
else:
    !git -C "$repo_path" pull

if repo_path in sys.path:
    sys.path.remove(repo_path)

sys.path.insert(0, repo_path)

for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

importlib.invalidate_caches()


estimator_path = Path(repo_path) / "src" / "estimator.py"
print("Importing from:", estimator_path)

from src.estimator import *

print("Successfully imported estimator functions.")

Cloning into '/content/Quant-Finance-Summer-2026'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 65 (delta 19), reused 36 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (65/65), 578.21 KiB | 38.55 MiB/s, done.
Resolving deltas: 100% (19/19), done.
Importing from: /content/Quant-Finance-Summer-2026/src/estimator.py
Successfully imported estimator functions.


#Experiment on market data
#Download 1-minute data, compute realized volatility for different window sizes, estimate H for each RV path, and return a table.

In [4]:
def download_1m_data_chunked(
    ticker="AAPL",
    start=None,
    end=None,
    max_days_back=29,
    chunk_days=7,
    regular_hours=True,
    timezone="America/New_York"
):
    if end is None:
        end_ts = pd.Timestamp.now(tz=timezone)
    else:
        end_ts = pd.Timestamp(end)
        if end_ts.tzinfo is None:
            end_ts = end_ts.tz_localize(timezone)

    if start is None:
        start_ts = end_ts - pd.Timedelta(days=max_days_back)
    else:
        start_ts = pd.Timestamp(start)
        if start_ts.tzinfo is None:
            start_ts = start_ts.tz_localize(timezone)

    # Warning if asking for too old 1-minute data
    oldest_allowed = pd.Timestamp.now(tz=timezone) - pd.Timedelta(days=60)

    if start_ts < oldest_allowed:
        print("Warning:")
        print("You are asking for 1-minute data older than about 60 days.")
        print("Yahoo/yfinance may return empty data for the older part.")
        print("Requested start:", start_ts)
        print("Approx oldest allowed:", oldest_allowed)

    all_chunks = []
    current_start = start_ts

    while current_start < end_ts:
        current_end = min(current_start + pd.Timedelta(days=chunk_days), end_ts)

        print(f"Downloading {ticker}: {current_start.date()} to {current_end.date()}")

        data = yf.download(
            ticker,
            start=current_start.strftime("%Y-%m-%d"),
            end=current_end.strftime("%Y-%m-%d"),
            interval="1m",
            auto_adjust=True,
            progress=False
        )

        if not data.empty:
            # Handle MultiIndex columns
            if isinstance(data.columns, pd.MultiIndex):
                close = data["Close"].iloc[:, 0]
            else:
                close = data["Close"]

            close = close.dropna()

            if len(close) > 0:
                all_chunks.append(close)

        current_start = current_end

    if len(all_chunks) == 0:
        raise ValueError(
            f"No 1-minute data downloaded for {ticker}. "
            "Try a more recent start date or a shorter period."
        )

    close = pd.concat(all_chunks)
    close = close[~close.index.duplicated(keep="first")]
    close = close.sort_index()
    close = close.dropna()

    # Convert timezone if needed
    if close.index.tz is None:
        close.index = close.index.tz_localize("UTC").tz_convert(timezone)
    else:
        close.index = close.index.tz_convert(timezone)

    if regular_hours:
        close = close.between_time("09:30", "16:00")

    close.name = ticker

    return close


def compute_realized_volatility_from_prices(
    prices,
    window_minutes=30,
    overlap=True,
    scale="raw"
):

    prices = prices.dropna()
    log_prices = np.log(prices)

    m = int(window_minutes)

    if m < 2:
        raise ValueError("window_minutes must be at least 2.")

    step = 1 if overlap else m

    rv_values = []
    rv_times = []

    for date, logp_day in log_prices.groupby(log_prices.index.date):

        returns = logp_day.diff().dropna()

        if len(returns) < m:
            continue

        for end in range(m, len(returns) + 1, step):
            window_returns = returns.iloc[end - m:end].values

            rv = np.sqrt(np.sum(window_returns ** 2))

            if scale == "per_minute":
                rv = rv / np.sqrt(m)
            elif scale != "raw":
                raise ValueError("scale must be 'raw' or 'per_minute'.")

            rv_values.append(rv)
            rv_times.append(returns.index[end - 1])

    rv = pd.Series(rv_values, index=pd.DatetimeIndex(rv_times))
    rv.name = f"RV_{window_minutes}min"

    return rv



def estimate_H_for_market_RV_windows(
    ticker="AAPL",
    start=None,
    end=None,
    windows_minutes=[2, 5, 10, 15, 30, 60],
    overlap=True,
    scale="raw",
    K_est=None,
    regular_hours=True,
    make_plots=False
):

    prices = download_1m_data_chunked(
        ticker=ticker,
        start=start,
        end=end,
        regular_hours=regular_hours
    )

    results = []
    rv_dict = {}

    for window in windows_minutes:

        rv = compute_realized_volatility_from_prices(
            prices=prices,
            window_minutes=window,
            overlap=overlap,
            scale=scale
        )

        rv_dict[window] = rv

        if len(rv) < 20:
            H_hat = np.nan
        else:
            H_hat = estimate_H(rv.values, K=K_est, T=1.0)

        results.append({
            "Ticker": ticker,
            "Window minutes": window,
            "Overlap": overlap,
            "Length RV": len(rv),
            "H_hat_RV": H_hat,
            "Mean RV": rv.mean(),
            "Std RV": rv.std()
        })

    results_df = pd.DataFrame(results)

    return results_df, prices, rv_dict

def market_RV_H_table_to_latex(results_df, digits=4):
    """
    Convert market realized-volatility H estimates to LaTeX.
    """

    rows = ""

    for _, row in results_df.iterrows():
        rows += (
            f'{int(row["Window minutes"])} & '
            f'{int(row["Length RV"])} & '
            f'{row["H_hat_RV"]:.{digits}f} \\\\\n'
        )

    latex = rf"""
\begin{{table}}[H]
\centering
\begin{{tabular}}{{cccc}}
\hline
RV window (minutes) & Length of RV path & $\widehat{{H}}(RV)$ \\
\hline
{rows}\hline
\end{{tabular}}
\caption{{Estimated roughness index of realized volatility for 1-minute market data. Realized volatility is computed using overlapping windows of log returns.}}
\end{{table}}
"""
    return latex


results_df, prices, rv_dict = estimate_H_for_market_RV_windows(
    ticker="AAPL",
    start=None,
    end=None,
    windows_minutes=[2, 5, 10, 15, 30, 60],
    overlap=True,
    scale="raw",
    K_est=None,
    regular_hours=True,
    make_plots=False
)

print(results_df)

print("\nLaTeX table:")
print(market_RV_H_table_to_latex(results_df, digits=4))

  Ticker  Window minutes  Overlap  Length RV  H_hat_RV   Mean RV    Std RV
0   AAPL               2     True       7372  0.083333  0.000841  0.000739
1   AAPL               5     True       7315  0.171404  0.001425  0.000978
2   AAPL              10     True       7220  0.286833  0.002041  0.001192
3   AAPL              15     True       7125  0.374841  0.002494  0.001320
4   AAPL              30     True       6840  0.505677  0.003486  0.001589
5   AAPL              60     True       6270  0.641295  0.004843  0.001917

LaTeX table:

\begin{table}[H]
\centering
\begin{tabular}{cccc}
\hline
RV window (minutes) & Length of RV path & $\widehat{H}(RV)$ \\
\hline
2 & 7372 & 0.0833 \\
5 & 7315 & 0.1714 \\
10 & 7220 & 0.2868 \\
15 & 7125 & 0.3748 \\
30 & 6840 & 0.5057 \\
60 & 6270 & 0.6413 \\
\hline
\end{tabular}
\caption{Estimated roughness index of realized volatility for 1-minute market data. Realized volatility is computed using overlapping windows of log returns.}
\end{table}



In [5]:
print("Downloaded price data summary")
print("-----------------------------")
print("Ticker:", prices.name)
print("Length of downloaded 1-minute prices:", len(prices))
print("Start:", prices.index.min())
print("End:", prices.index.max())
print("Trading days:", prices.index.normalize().nunique())

print("\nRealized-volatility path lengths")
print("--------------------------------")
for window, rv in rv_dict.items():
    print(f"RV window {window:>2} minutes: length = {len(rv)}")

Downloaded price data summary
-----------------------------
Ticker: AAPL
Length of downloaded 1-minute prices: 7410
Start: 2026-06-11 09:30:00-04:00
End: 2026-07-09 15:59:00-04:00
Trading days: 19

Realized-volatility path lengths
--------------------------------
RV window  2 minutes: length = 7372
RV window  5 minutes: length = 7315
RV window 10 minutes: length = 7220
RV window 15 minutes: length = 7125
RV window 30 minutes: length = 6840
RV window 60 minutes: length = 6270
